In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [10]:
pip install peft

Note: you may need to restart the kernel to use updated packages.


In [11]:
!pip uninstall -y torchao


In [12]:
!pip install -U peft

In [13]:
"""
Smart MCQ Solver Challenge - Full Rubric Pipeline
Roll number: 23f2004250

Implements the required 5-part rubric:
  1. LightGBM        -> TF-IDF + statistical/engineered features   (classical baseline)
  2. From-scratch MLP -> trained on TF-IDF vectors, pure PyTorch, no pretrained weights
  3. Pretrained       -> fine-tuned RoBERTa on (prompt + option) pairs
  4. Bonus/unique     -> LoRA-tuned RoBERTa (parameter-efficient fine-tuning via PEFT)
  5. Ensemble          -> weighted average of the best 2-3 models' probabilities -> top-3

Run on Kaggle with GPU enabled (Settings -> Accelerator -> GPU T4/P100).

Paths assume the Kaggle competition environment:
  /kaggle/input/competitions/smart-mcq-solver-challenge/mcq_train_dataset.csv
  /kaggle/input/competitions/smart-mcq-solver-challenge/mcq_test_dataset.csv
Output goes to /kaggle/working/submission.csv
"""

import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD

RANDOM_STATE = 42
OPTIONS = ["A", "B", "C", "D", "E"]
N_FOLDS = 2

# ---- toggle which environment you're running in --------------------------
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
    OUT_DIR = "/kaggle/working"
else:
    DATA_DIR = "/mnt/user-data/uploads"
    OUT_DIR = "/mnt/user-data/outputs"

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SUBMISSION_PATH = f"{OUT_DIR}/submission.csv"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================================
# 0. Shared data utilities
# ============================================================================
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    return train, test


def to_long(df, is_train):
    rows = []
    for _, r in df.iterrows():
        for opt in OPTIONS:
            rows.append({
                "id": r["id"],
                "prompt": r["prompt"],
                "option_letter": opt,
                "option_text": r[opt],
                "label": int(is_train and r["answer"] == opt),
            })
    return pd.DataFrame(rows)


def word_set(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


def map_at_3(y_true_letters, ranked_preds):
    scores = []
    for true, preds in zip(y_true_letters, ranked_preds):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = 1.0 / (i + 1)
                break
        scores.append(s)
    return float(np.mean(scores))


def proba_to_ranked_letters(long_df, proba):
    tmp = long_df.copy()
    tmp["proba"] = proba
    ranked = (
        tmp.sort_values(["id", "proba"], ascending=[True, False])
        .groupby("id")["option_letter"]
        .apply(list)
    )
    return ranked


def evaluate(name, long_df, id_order, truth_series, proba):
    ranked = proba_to_ranked_letters(long_df, proba).reindex(id_order)
    score = map_at_3(truth_series.reindex(id_order).tolist(), ranked.tolist())
    print(f"[{name}] MAP@3 = {score:.4f}")
    return score


# ============================================================================
# 1. Feature engineering (shared by LightGBM and the from-scratch MLP)
# ============================================================================
def add_engineered_features(long_df):
    long_df = long_df.copy()
    long_df["opt_len_chars"] = long_df["option_text"].astype(str).str.len()
    long_df["opt_len_words"] = long_df["option_text"].astype(str).str.split().str.len()
    long_df["prompt_len_chars"] = long_df["prompt"].astype(str).str.len()
    long_df["prompt_len_words"] = long_df["prompt"].astype(str).str.split().str.len()

    grp_len = long_df.groupby("id")["opt_len_chars"]
    long_df["len_rank_pct"] = long_df.groupby("id")["opt_len_chars"].rank(pct=True)
    long_df["len_minus_mean"] = long_df["opt_len_chars"] - grp_len.transform("mean")
    long_df["len_zscore"] = long_df["len_minus_mean"] / (grp_len.transform("std") + 1e-6)
    long_df["is_longest"] = (long_df["opt_len_chars"] == grp_len.transform("max")).astype(int)
    long_df["is_shortest"] = (long_df["opt_len_chars"] == grp_len.transform("min")).astype(int)

    for opt in OPTIONS:
        long_df[f"is_opt_{opt}"] = (long_df["option_letter"] == opt).astype(int)

    prompt_words = long_df["prompt"].apply(word_set)
    option_words = long_df["option_text"].apply(word_set)
    overlap = [len(p & o) for p, o in zip(prompt_words, option_words)]
    long_df["prompt_overlap_count"] = overlap
    long_df["prompt_overlap_ratio"] = [c / (len(o) + 1e-6) for c, o in zip(overlap, option_words)]

    long_df["_wordset"] = option_words
    other_mean, other_max = [], []
    for qid, group in long_df.groupby("id"):
        sets = group["_wordset"].tolist()
        for i in range(len(sets)):
            sims = []
            for j in range(len(sets)):
                if i == j:
                    continue
                a, b = sets[i], sets[j]
                u = len(a | b)
                sims.append(len(a & b) / u if u else 0.0)
            other_mean.append(np.mean(sims) if sims else 0.0)
            other_max.append(np.max(sims) if sims else 0.0)
    long_df["other_overlap_mean"] = other_mean
    long_df["other_overlap_max"] = other_max
    long_df.drop(columns=["_wordset"], inplace=True)

    long_df["num_commas"] = long_df["option_text"].astype(str).str.count(",")
    long_df["num_digits"] = long_df["option_text"].astype(str).str.count(r"\d")
    long_df["has_negation"] = long_df["option_text"].astype(str).str.contains(
        r"\bnot\b|\bno\b|\bnever\b|\bcannot\b|\bdoes not\b|\bdon't\b", case=False
    ).astype(int)
    return long_df


ENGINEERED_FEATURES = [
    "opt_len_chars", "opt_len_words", "prompt_len_chars", "prompt_len_words",
    "len_rank_pct", "len_minus_mean", "len_zscore", "is_longest", "is_shortest",
    "is_opt_A", "is_opt_B", "is_opt_C", "is_opt_D", "is_opt_E",
    "prompt_overlap_count", "prompt_overlap_ratio",
    "other_overlap_mean", "other_overlap_max",
    "num_commas", "num_digits", "has_negation",
]


def build_tfidf_features(train_long, test_long, n_svd=64):
    """Fit TF-IDF on prompt+option text, add cosine sim to prompt,
    and return dense SVD-compressed embeddings for the option text
    (used as extra numeric features / as MLP input)."""
    tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words="english")
    all_text = pd.concat(
        [train_long["prompt"], train_long["option_text"],
         test_long["prompt"], test_long["option_text"]], axis=0
    ).astype(str)
    tfidf.fit(all_text)

    def sim_to_prompt(df):
        p = tfidf.transform(df["prompt"].astype(str))
        o = tfidf.transform(df["option_text"].astype(str))
        return np.array([cosine_similarity(p[i], o[i])[0, 0] for i in range(df.shape[0])])

    train_long = train_long.copy()
    test_long = test_long.copy()
    train_long["tfidf_sim_to_prompt"] = sim_to_prompt(train_long)
    test_long["tfidf_sim_to_prompt"] = sim_to_prompt(test_long)
    train_long["tfidf_sim_rank_pct"] = train_long.groupby("id")["tfidf_sim_to_prompt"].rank(pct=True)
    test_long["tfidf_sim_rank_pct"] = test_long.groupby("id")["tfidf_sim_to_prompt"].rank(pct=True)

    # SVD-compressed dense embedding of option text -> used as MLP input
    svd = TruncatedSVD(n_components=n_svd, random_state=RANDOM_STATE)
    train_opt_vecs = tfidf.transform(train_long["option_text"].astype(str))
    test_opt_vecs = tfidf.transform(test_long["option_text"].astype(str))
    svd.fit(train_opt_vecs)
    train_svd = svd.transform(train_opt_vecs)
    test_svd = svd.transform(test_opt_vecs)

    return train_long, test_long, train_svd, test_svd


# ============================================================================
# MODEL 1: LightGBM  (TF-IDF-derived + statistical features)
# ============================================================================
def run_lightgbm(train_long, test_long, groups, id_order_train, id_order_test, truth):
    import lightgbm as lgb

    feats = ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]
    X, y = train_long[feats], train_long["label"]
    X_test = test_long[feats]

    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    params = dict(
        objective="binary", metric="auc", num_leaves=31, learning_rate=0.03,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=15, n_estimators=2000, random_state=RANDOM_STATE, verbosity=-1,
    )

    for tr_idx, va_idx in gkf.split(X, y, groups):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        test_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS

    evaluate("LightGBM (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred


# ============================================================================
# MODEL 2: From-scratch MLP (PyTorch, trained on TF-IDF/SVD features)
# ============================================================================
def run_mlp(train_svd, test_svd, train_extra, test_extra, train_long, test_long,
            groups, id_order_train, id_order_test, truth):
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_all = np.concatenate([train_svd, train_extra], axis=1).astype(np.float32)
    X_test_all = np.concatenate([test_svd, test_extra], axis=1).astype(np.float32)
    y_all = train_long["label"].values.astype(np.float32)

    mean, std = X_all.mean(0, keepdims=True), X_all.std(0, keepdims=True) + 1e-6
    X_all = (X_all - mean) / std
    X_test_all = (X_test_all - mean) / std

    class TabDataset(Dataset):
        def __init__(self, X, y=None):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            if self.y is None:
                return self.X[idx]
            return self.X[idx], self.y[idx]

    class MLP(nn.Module):
        """Simple from-scratch feed-forward network — no pretrained weights."""
        def __init__(self, in_dim, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)

    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        model = MLP(X_all.shape[1]).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()

        train_loader = DataLoader(
            TabDataset(X_all[tr_idx], y_all[tr_idx]), batch_size=256, shuffle=True
        )
        best_val = -1
        patience, bad_epochs = 5, 0
        best_state = None

        for epoch in range(60):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                va_logits = model(torch.tensor(X_all[va_idx]).to(device)).cpu().numpy()
            va_proba = 1 / (1 + np.exp(-va_logits))
            ranked = proba_to_ranked_letters(
                train_long.iloc[va_idx], va_proba
            )
            va_ids = train_long.iloc[va_idx]["id"].unique()
            score = map_at_3(truth.reindex(va_ids).tolist(), ranked.reindex(va_ids).tolist())

            if score > best_val:
                best_val, bad_epochs = score, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[va_idx] = 1 / (1 + np.exp(-model(
                torch.tensor(X_all[va_idx]).to(device)).cpu().numpy()))
            test_pred += (1 / (1 + np.exp(-model(
                torch.tensor(X_test_all).to(device)).cpu().numpy()))) / N_FOLDS
        print(f"MLP fold {fold}: best val MAP@3 = {best_val:.4f}")

    evaluate("From-scratch MLP (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred


# ============================================================================
# MODEL 3: Fine-tuned pretrained transformer (RoBERTa) on prompt+option
# ============================================================================
def run_transformer(train_long, test_long, groups, id_order_train, id_order_test,
                     truth, model_name="roberta-base", use_lora=False, epochs=2):
    """Fine-tunes (optionally with LoRA) a pretrained transformer as a
    binary sequence-classification head over (prompt, option) pairs."""
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class PairDataset(Dataset):
        def __init__(self, df, labels=None):
            self.prompt = df["prompt"].astype(str).tolist()
            self.option = df["option_text"].astype(str).tolist()
            self.labels = labels

        def __len__(self):
            return len(self.prompt)

        def __getitem__(self, idx):
            item = {"prompt": self.prompt[idx], "option": self.option[idx]}
            if self.labels is not None:
                item["label"] = self.labels[idx]
            return item

    def collate(batch):
        prompts = [b["prompt"] for b in batch]
        options = [b["option"] for b in batch]
        enc = tokenizer(
            prompts, options, truncation=True, padding=True,
            max_length=256, return_tensors="pt",
        )
        if "label" in batch[0]:
            enc["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.float32)
        return enc

    gkf = GroupKFold(n_splits=N_FOLDS)
    y_all = train_long["label"].values.astype(np.float32)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    # For speed, transformer fine-tuning typically only needs 1 fold's worth
    # of held-out validation on Kaggle's free GPU time budget; loop below
    # still supports full K-fold if you have the compute/time.
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_long, y_all, groups)):
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

        if use_lora:
            from peft import LoraConfig, get_peft_model, TaskType
            lora_cfg = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=16, lora_alpha=32, lora_dropout=0.1,
                target_modules=["query", "value"],  # attention proj layers
            )
            model = get_peft_model(model, lora_cfg)
            model.print_trainable_parameters()

        model.to(device)

        train_loader = DataLoader(
            PairDataset(train_long.iloc[tr_idx].reset_index(drop=True), y_all[tr_idx]),
            batch_size=16, shuffle=True, collate_fn=collate,
        )
        va_loader = DataLoader(
            PairDataset(train_long.iloc[va_idx].reset_index(drop=True)),
            batch_size=32, shuffle=False, collate_fn=collate,
        )
        test_loader = DataLoader(
            PairDataset(test_long.reset_index(drop=True)),
            batch_size=32, shuffle=False, collate_fn=collate,
        )

        opt = torch.optim.AdamW(model.parameters(), lr=2e-5 if not use_lora else 1e-4)
        loss_fn = nn.BCEWithLogitsLoss()

        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                labels = batch.pop("labels")
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = labels.to(device)
                opt.zero_grad()
                logits = model(**batch).logits.squeeze(-1)
                loss = loss_fn(logits, labels)
                loss.backward()
                opt.step()
            print(f"  [{model_name}{'+LoRA' if use_lora else ''}] fold {fold} epoch {epoch} done")

        def predict(loader):
            model.eval()
            preds = []
            with torch.no_grad():
                for batch in loader:
                    batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                    logits = model(**batch).logits.squeeze(-1)
                    preds.append(torch.sigmoid(logits).cpu().numpy())
            return np.concatenate(preds)

        oof[va_idx] = predict(va_loader)
        test_pred += predict(test_loader) / N_FOLDS

        del model
        torch.cuda.empty_cache()

    tag = f"{model_name}{' + LoRA' if use_lora else ''}"
    evaluate(f"{tag} (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred


# ============================================================================
# Ensemble: weighted average of chosen models' probabilities -> top-3
# ============================================================================
def ensemble_and_submit(long_df, id_order, model_probas: dict, weights: dict, out_path):
    """model_probas: {name: proba_array}; weights: {name: float}"""
    combined = np.zeros(len(long_df))
    total_w = sum(weights.values())
    for name, proba in model_probas.items():
        combined += (weights[name] / total_w) * proba

    ranked = proba_to_ranked_letters(long_df, combined).reindex(id_order)
    return combined, ranked


def build_submission(id_col, ranked_letters, out_path):
    sub = pd.DataFrame({
        "ID": id_col,
        "Prediction": [" ".join(letters[:3]) for letters in ranked_letters],
    })
    sub.to_csv(out_path, index=False)
    print(f"Saved submission -> {out_path}")
    print(sub.head())
    return sub


# ============================================================================
# MAIN
# ============================================================================
def main(run_transformers=True, run_lora=True):
    train_wide, test_wide = load_data()
    truth = train_wide.set_index("id")["answer"]

    train_long = to_long(train_wide, is_train=True)
    test_long = to_long(test_wide, is_train=False)

    print("Adding engineered features...")
    train_long = add_engineered_features(train_long)
    test_long = add_engineered_features(test_long)

    print("Building TF-IDF / SVD features...")
    train_long, test_long, train_svd, test_svd = build_tfidf_features(train_long, test_long)

    groups = train_long["id"]
    id_order_train = train_wide["id"]
    id_order_test = test_wide["id"]

    results_oof, results_test = {}, {}

    # ---- 1. LightGBM -------------------------------------------------------
    print("\n=== Model 1: LightGBM ===")
    oof, test_pred = run_lightgbm(train_long, test_long, groups, id_order_train, id_order_test, truth)
    results_oof["lgbm"], results_test["lgbm"] = oof, test_pred

    # ---- 2. From-scratch MLP -----------------------------------------------
    print("\n=== Model 2: From-scratch MLP ===")
    train_extra = train_long[ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]].values
    test_extra = test_long[ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]].values
    oof, test_pred = run_mlp(
        train_svd, test_svd, train_extra, test_extra,
        train_long, test_long, groups, id_order_train, id_order_test, truth,
    )
    results_oof["mlp"], results_test["mlp"] = oof, test_pred

    # ---- 3 & 4. Transformers (pretrained fine-tune + LoRA bonus) ----------
    if run_transformers:
        print("\n=== Model 3: Fine-tuned RoBERTa ===")
        oof, test_pred = run_transformer(
            train_long, test_long, groups, id_order_train, id_order_test, truth,
            model_name="roberta-base", use_lora=False, epochs=2,
        )
        results_oof["roberta"], results_test["roberta"] = oof, test_pred

        if run_lora:
            print("\n=== Model 4 (Bonus): LoRA-tuned RoBERTa ===")
            oof, test_pred = run_transformer(
                train_long, test_long, groups, id_order_train, id_order_test, truth,
                model_name="roberta-base", use_lora=True, epochs=3,
            )
            results_oof["roberta_lora"], results_test["roberta_lora"] = oof, test_pred

    # ---- 5. Ensemble: pick best 2-3 models by OOF MAP@3 --------------------
    print("\n=== Scoring individual models to pick ensemble members ===")
    scores = {}
    for name, oof in results_oof.items():
        ranked = proba_to_ranked_letters(train_long, oof).reindex(id_order_train)
        scores[name] = map_at_3(truth.reindex(id_order_train).tolist(), ranked.tolist())
        print(f"  {name}: {scores[name]:.4f}")

    top_models = sorted(scores, key=scores.get, reverse=True)[:3]
    print(f"\nEnsembling top models: {top_models}")
    weights = {name: scores[name] for name in top_models}  # weight ~ OOF score

    oof_combo, _ = ensemble_and_submit(
        train_long, id_order_train,
        {n: results_oof[n] for n in top_models}, weights, None,
    )
    ensemble_score = map_at_3(
        truth.reindex(id_order_train).tolist(),
        proba_to_ranked_letters(train_long, oof_combo).reindex(id_order_train).tolist(),
    )
    print(f"Ensemble OOF MAP@3 = {ensemble_score:.4f}")

    _, test_ranked = ensemble_and_submit(
        test_long, id_order_test,
        {n: results_test[n] for n in top_models}, weights, None,
    )
    build_submission(id_order_test, test_ranked, SUBMISSION_PATH)

    return scores, ensemble_score


if __name__ == "__main__":
    # On limited compute (e.g. CPU-only / quick local test), set
    # run_transformers=False to only run the LightGBM + MLP baseline pair.
    main(run_transformers=True, run_lora=True)

Adding engineered features...
Building TF-IDF / SVD features...

=== Model 1: LightGBM ===
[LightGBM (OOF)] MAP@3 = 0.9928

=== Model 2: From-scratch MLP ===
MLP fold 0: best val MAP@3 = 0.9950
MLP fold 1: best val MAP@3 = 0.9867
[From-scratch MLP (OOF)] MAP@3 = 0.9908

=== Model 3: Fine-tuned RoBERTa ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [roberta-base] fold 0 epoch 0 done
  [roberta-base] fold 0 epoch 1 done


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,181,185 || all params: 125,827,586 || trainable%: 0.9387
  [roberta-base+LoRA] fold 0 epoch 0 done
  [roberta-base+LoRA] fold 0 epoch 1 done
  [roberta-base+LoRA] fold 0 epoch 2 done


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,181,185 || all params: 125,827,586 || trainable%: 0.9387
  [roberta-base+LoRA] fold 1 epoch 0 done
  [roberta-base+LoRA] fold 1 epoch 1 done
  [roberta-base+LoRA] fold 1 epoch 2 done
[roberta-base + LoRA (OOF)] MAP@3 = 0.6595

=== Scoring individual models to pick ensemble members ===
  lgbm: 0.9928
  mlp: 0.9908
  roberta: 0.6032
  roberta_lora: 0.6595

Ensembling top models: ['lgbm', 'mlp', 'roberta_lora']
Ensemble OOF MAP@3 = 0.9932
Saved submission -> /kaggle/working/submission.csv
   ID Prediction
0   1      A B C
1   2      B E C
2   3      B E C
3   4      E A C
4   5      C A B
